### Week 5 Day 4

AutoGen Core - Distributed

I'm only going to give a Teaser of this!!

Partly because I'm unsure how relevant it is to you. If you'd like me to add more content for this, please do let me know..

In [1]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.langchain import LangChainToolAdapter
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import Tool
from IPython.display import display, Markdown

from dotenv import load_dotenv

load_dotenv(override=True)

ALL_IN_ONE_WORKER = False

### Start with our Message class

In [3]:

@dataclass
class Message:
    content: str

### And now - a host for our distributed runtime

In [4]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntimeHost

host = GrpcWorkerAgentRuntimeHost(address="localhost:50051")
host.start() 

### Let's reintroduce a tool

In [5]:
serper = GoogleSerperAPIWrapper()
langchain_serper =Tool(name="internet_search", func=serper.run, description="Useful for when you need to search the internet")
autogen_serper = LangChainToolAdapter(langchain_serper)

In [6]:
instruction1 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons in favor of choosing AutoGen; the pros of AutoGen."

instruction2 = "To help with a decision on whether to use AutoGen in a new AI Agent project, \
please research and briefly respond with reasons against choosing AutoGen; the cons of Autogen."

judge = "You must make a decision on whether to use AutoGen for a project. \
Your research team has come up with the following reasons for and against. \
Based purely on the research from your team, please respond with your decision and brief rationale."

### And make some Agents

In [7]:
class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client, tools=[autogen_serper], reflect_on_tool_use=True)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Judge(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent(name, model_client=model_client)
        
    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        message1 = Message(content=instruction1)
        message2 = Message(content=instruction2)
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message1, inner_1)
        response2 = await self.send_message(message2, inner_2)
        result = f"## Pros of AutoGen:\n{response1.content}\n\n## Cons of AutoGen:\n{response2.content}\n\n"
        judgement = f"{judge}\n{result}Respond with your decision and brief explanation"
        message = TextMessage(content=judgement, source="user")
        response = await self._delegate.on_messages([message], ctx.cancellation_token)
        return Message(content=result + "\n\n## Decision:\n\n" + response.chat_message.content)


In [8]:
from autogen_ext.runtimes.grpc import GrpcWorkerAgentRuntime

if ALL_IN_ONE_WORKER:

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()

    await Player1Agent.register(worker, "player1", lambda: Player1Agent("player1"))
    await Player2Agent.register(worker, "player2", lambda: Player2Agent("player2"))
    await Judge.register(worker, "judge", lambda: Judge("judge"))

    agent_id = AgentId("judge", "default")

else:

    worker1 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker1.start()
    await Player1Agent.register(worker1, "player1", lambda: Player1Agent("player1"))

    worker2 = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker2.start()
    await Player2Agent.register(worker2, "player2", lambda: Player2Agent("player2"))

    worker = GrpcWorkerAgentRuntime(host_address="localhost:50051")
    await worker.start()
    await Judge.register(worker, "judge", lambda: Judge("judge"))
    agent_id = AgentId("judge", "default")




In [9]:
response = await worker.send_message(Message(content="Go!"), agent_id)

In [10]:
display(Markdown(response.content))

## Pros of AutoGen:
Here are some pros of using AutoGen in your AI Agent project:

1. **Modular Architecture**: AutoGen features a modular structure that separates perception, reasoning, and action components. This makes it easier to develop, test, and maintain AI agents.

2. **Multi-Agent Collaboration**: It excels in facilitating communication and collaboration between multiple agents, which is beneficial for complex problem-solving tasks.

3. **Efficiency and Speed**: AutoGen significantly improves data processing speed and efficiency, which is critical in applications requiring real-time responses.

4. **Ease of Use**: The framework is user-friendly, streamlining the development process and reducing the effort required for drafting and deploying tasks.

5. **Customizability**: Users can create agents that are customizable and adaptable to various scenarios, allowing for flexible solutions tailored to specific needs.

6. **Integration Capability**: AutoGen can easily integrate with various LLMs (Large Language Models), tools, and even incorporate human input to enhance the agent's capabilities.

7. **Built-in Features**: It comes with built-in features like retry logic, caching, model fallback, and goal-setting, which simplify workflow management and enhance reliability.

8. **Open Source**: As an open-source framework, AutoGen allows for community contributions and improvements, ensuring it stays current with the latest advancements in AI technology.

These advantages make AutoGen a compelling option for projects that require efficient, scalable, and customizable AI agent solutions. 

TERMINATE

## Cons of AutoGen:
Here are some cons of using AutoGen for AI Agent projects:

1. **Higher Technical Barrier**: Implementing AutoGen requires a deep understanding of how multiple agents interact, which can be challenging for teams without the necessary expertise.

2. **Complex Setup Requirements**: Getting started with AutoGen may take considerable time and effort, particularly if the team lacks familiarity with its framework.

3. **Agent Coordination Complexity**: Managing and coordinating multiple agents can introduce complexities, making it harder to ensure smooth operation and communication between them.

4. **Debugging Multi-Agent Systems**: Debugging issues in a multi-agent system can be significantly more complicated than in single-agent configurations, leading to potential delays in development.

5. **Production Deployment Challenges**: Transitioning to a production environment can have its own set of challenges, which may require additional resources and careful planning.

These factors may lead some teams to reconsider using AutoGen for their AI Agent projects. 

TERMINATE



## Decision:

Based on the research provided, I recommend using AutoGen for the project. The pros outweigh the cons since its modular architecture, efficiency, and ease of use can significantly enhance the development process and facilitate effective multi-agent collaboration. While the technical barriers and complexity in setup are valid concerns, the long-term benefits of customizability, integration capability, and built-in features suggest that these challenges can be effectively managed. Overall, AutoGen appears to be a robust choice for developing scalable and efficient AI agents.

TERMINATE

In [11]:
await worker.stop()
if not ALL_IN_ONE_WORKER:
    await worker1.stop()
    await worker2.stop()

In [12]:
await host.stop()